# Amazon Customer Feedback Analysis and Automated Response
### Imarticus Data Science Internship Assessment — Python Foundations & Gen AI

This notebook analyzes the supplied `amazon.csv` dataset and uses the Groq API to draft responses to three critical reviews.

**Columns used:** `product_name`, `rating`, `review_title`, `review_content`, `user_name`, `product_id`, and `review_id`.

Workflow: data cleaning → rule-based critical review filtering (ratings 1–3) → complaint keyword analysis → select three detailed negative reviews → generate response email drafts.

> Note: the assessment gives 1–2 star reviews as an example. This notebook uses ratings of 3 or lower to include a broader set of critical feedback. API-generated drafts require a working Groq API key; fallback drafts are labeled separately.

## 1. Import libraries

In [ ]:
import os
import re
from collections import Counter
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 180)

## 2. Load the Amazon dataset

In [ ]:
DATA_PATH = Path("amazon.csv")
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("Columns:", df.columns.tolist())
display(df.head(3))

## 3. Inspect and clean the data

In [ ]:
required_columns = ["product_name", "rating", "review_content"]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Expected columns not found in amazon.csv: {missing_columns}")

print("Missing values before cleaning:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))

df = df.dropna(subset=["rating", "review_content"]).copy()
for col in ["product_name", "review_title", "user_name", "product_id", "review_id"]:
    if col in df.columns:
        df[col] = df[col].fillna("Not provided")

# The rating column is stored as text in this dataset.
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df = df.dropna(subset=["rating"]).copy()
df = df[df["rating"].between(1, 5)].copy()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    return re.sub(r"\s+", " ", text).strip()

df["clean_review"] = (
    df["review_title"].astype(str) + " " + df["review_content"].astype(str)
).apply(clean_text)

print("Rows after cleaning:", len(df))
print("Missing values after cleaning:")
display(df[["rating", "review_title", "review_content", "clean_review"]].isna().sum().to_frame("missing_values"))
display(df[["product_name", "rating", "review_title", "review_content", "clean_review"]].head(3))

## 4. Rule-based filtering of critical reviews (no machine learning)

In [ ]:
# The assessment gives ratings of 3 stars or lower as examples of critical reviews.
critical_reviews = df[df["rating"] <= 3].copy()
critical_reviews = critical_reviews.sort_values("rating", ascending=True).reset_index(drop=True)

total_reviews = len(df)
critical_count = len(critical_reviews)
critical_pct = (critical_count / total_reviews * 100) if total_reviews else 0

print(f"Total usable reviews: {total_reviews:,}")
print(f"Critical reviews (rating <= 3): {critical_count:,}")
print(f"Critical review percentage: {critical_pct:.2f}%")
display(critical_reviews[["product_name", "rating", "review_title", "review_content"]].head(10))

## 5. Complaint keyword analysis

In [ ]:
complaint_keywords = {
    "broken", "damaged", "late", "delay", "slow", "rude", "unhelpful",
    "refund", "leak", "leaks", "stopped", "working", "cracked", "wrong",
    "poor", "disappointing", "defective", "issue", "problem", "waste",
    "bad", "worst", "quality", "durable", "charging", "charge",
    "disconnect", "replacement", "return", "failure", "fail", "faulty"
}

keyword_counts = Counter()
for review in critical_reviews["clean_review"]:
    # Count each listed keyword at most once per review.
    keyword_counts.update(set(review.split()).intersection(complaint_keywords))

keyword_summary = pd.DataFrame(
    keyword_counts.most_common(15),
    columns=["complaint_keyword", "review_count"]
)
print("Most common complaint keywords in critical reviews:")
display(keyword_summary)

## 5. Complaint keyword analysis

The notebook displays a keyword-count table and a short calculated insight summary based on the cleaned dataset.

In [ ]:
# Prioritize lowest ratings, then longer review text as a simple detail heuristic.
selected_reviews = (
    critical_reviews.assign(review_length=critical_reviews["review_content"].astype(str).str.len())
    .sort_values(["rating", "review_length"], ascending=[True, False])
    .head(3)
    .copy()
    .reset_index(drop=True)
)

display(selected_reviews[[
    "product_name", "rating", "review_title", "review_content",
    "user_name", "product_id", "review_id"
]])

## 7. Generate personalized apology emails with Groq

This section uses the Groq API when `GROQ_API_KEY` is configured in `gem.env`. If the key is missing or an API request fails, it creates clearly labeled template drafts. Template drafts are not claimed to be AI-generated. Never hard-code your API key in the notebook.

### Groq API setup

Run `%pip install groq python-dotenv` if the packages are not installed. Create `gem.env` beside this notebook and add `GROQ_API_KEY=your_groq_api_key_here`. Keep this file private. Check the currently supported model IDs in the [Groq model documentation](https://console.groq.com/docs/models) if the configured model is unavailable.

In [ ]:
from dotenv import load_dotenv

# Load the key from gem.env in the same folder as this notebook.
load_dotenv("gem.env", override=True)
api_key = os.getenv("GROQ_API_KEY")


def make_fallback_email(row):
    customer = str(row.get("user_name", "Customer"))
    product = str(row.get("product_name", "product"))
    review = str(row.get("review_content", ""))
    return (
        f"Subject: We're sorry about your experience with {product}\n\n"
        f"Dear {customer},\n\n"
        f"We're sorry that your experience with {product} did not meet expectations. "
        f"We understand your concern: {review[:700]}\n\n"
        f"Our support team can review the details and help identify an appropriate next step. "
        f"Please contact customer support and include your product details so the issue can be investigated.\n\n"
        f"Thank you for sharing your feedback.\n\n"
        f"Kind regards,\nCustomer Support Team"
    )


def generate_email_groq(row, client):
    """Generate a personalized customer-support email using the Groq API."""
    prompt = f"""
Write a short, personalized, empathetic apology email responding to this negative product review.

Instructions:
- Acknowledge the customer's specific concern.
- Use a polite and professional tone.
- Do not invent facts or make unsupported promises.
- Do not claim a refund or replacement is approved.
- Ask the customer to contact support for further investigation.
- Include a subject line, greeting, body, and sign-off.
- Keep the email concise.

Product: {row.get('product_name', 'Not provided')}
Rating: {row.get('rating', 'Not provided')} out of 5
Review title: {row.get('review_title', 'Not provided')}
Review content: {row.get('review_content', 'Not provided')}
"""
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": "You are a professional retail customer support agent. Write empathetic, accurate customer emails."
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0.5,
        max_completion_tokens=500,
    )
    email_text = response.choices[0].message.content
    if not email_text or not email_text.strip():
        raise ValueError("Groq returned an empty response.")
    return email_text.strip()


# Select three reviews above; this loop preserves one output row per selected review.
generated_emails = []

if api_key:
    try:
        from groq import Groq
        client = Groq(api_key=api_key)
        for _, row in selected_reviews.iterrows():
            email_text = generate_email_groq(row, client)
            generated_emails.append({
                "product_id": row.get("product_id", ""),
                "review_id": row.get("review_id", ""),
                "source": "Groq API",
                "email": email_text,
            })
            print(f"Successfully generated email for review {row.get('review_id', '')}")
    except Exception as exc:
        print("Groq generation failed; using template fallback drafts for all three reviews.")
        print(f"Error type: {type(exc).__name__}. Check the API key, model access, quota, and network.")
        generated_emails = [
            {
                "product_id": row.get("product_id", ""),
                "review_id": row.get("review_id", ""),
                "source": "Template fallback",
                "email": make_fallback_email(row),
            }
            for _, row in selected_reviews.iterrows()
        ]
else:
    print("GROQ_API_KEY not found in gem.env. These are template drafts, not AI-generated outputs.")
    generated_emails = [
        {
            "product_id": row.get("product_id", ""),
            "review_id": row.get("review_id", ""),
            "source": "Template fallback",
            "email": make_fallback_email(row),
        }
        for _, row in selected_reviews.iterrows()
    ]

# Display the three response drafts in the notebook output.
for item in generated_emails:
    display(Markdown(f"### Review {item['review_id']} — {item['source']}"))
    display(Markdown(item["email"].replace("\n", "  \n").replace("\n", "  \n")))

print("\nEmail draft count:", len(generated_emails))
print("Groq-generated drafts:", sum(item["source"] == "Groq API" for item in generated_emails))
print("Template fallback drafts:", sum(item["source"] == "Template fallback" for item in generated_emails))


## 8. Export analysis outputs

The notebook saves the critical reviews, keyword summary, and three response drafts to CSV files. Run all cells with your Groq API key configured to populate the notebook outputs with the generated emails.

In [ ]:
critical_reviews.to_csv("critical_reviews.csv", index=False)
keyword_summary.to_csv("complaint_keyword_summary.csv", index=False)
pd.DataFrame(generated_emails).to_csv("generated_response_emails.csv", index=False)

print("Exported critical_reviews.csv")
print("Exported complaint_keyword_summary.csv")
print("Exported generated_response_emails.csv")

## 9. Conclusion

- This notebook uses the supplied Amazon dataset’s `rating`, `review_title`, and `review_content` columns.
- Reviews rated 3 or lower are treated as critical using a rule-based threshold; no machine learning is used.
- Complaint terms are counted with `collections.Counter`.
- Three low-rated, detailed reviews are selected for response drafting.
- Groq-generated emails require a valid `GROQ_API_KEY` in `gem.env`; fallback drafts are clearly labeled.
- Keyword matching is limited to the predefined keyword list.
- Review all drafts before sending them to real customers.